# RAG + Qwen: Ad Marketing Analysis

This notebook runs **Retrieval-Augmented Generation** using **Qwen2** and your ad datasets. Designed for **Google Colab** with GPU.

## 1. Install Dependencies

In [ ]:
!pip install -q pandas chromadb sentence-transformers transformers accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently 

In [ ]:
!pip install -q faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 53.1 MB/s eta 0:00:00


## 2. GPU Check & Data Paths

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import json
import pandas as pd
import torch

BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/intelligence_ai_marketing"

ADS_CSV = f"{BASE_DIR}/data/ads_with_strategies.csv"
INSIGHT_DIR = f"{BASE_DIR}/analytics/insights"

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

ads = pd.read_csv(ADS_CSV)
print(f"Loaded {len(ads)} ads from {ADS_CSV}")

with open(f"{INSIGHT_DIR}/strategy_insights.json", "r") as f:
    strategy_insights = json.load(f)

with open(f"{INSIGHT_DIR}/brand_strategy_patterns.json", "r") as f:
    brand_patterns = json.load(f)

with open(f"{INSIGHT_DIR}/visual_design_patterns.json", "r") as f:
    visual_patterns = json.load(f)

print("Analytics insights loaded")
print(ads.columns)

GPU available: True
GPU: Tesla T4
Loaded 2046 ads from /content/drive/MyDrive/Colab Notebooks/intelligence_ai_marketing/data/ads_with_strategies.csv
Analytics insights loaded
Index(['ad_id', 'json_key', 'image_path', 'competitor', 'all_categories',
       'all_categories_full', 'all_sentiments', 'all_sentiments_full',
       'objects_symbols', 'image_width', 'image_height', 'image_format',
       'ocr_text', 'ocr_word_count', 'ocr_confidence_avg', 'text_area_px',
       'image_area_px', 'text_image_ratio', 'layout_type', 'dominant_color_1',
       'dominant_color_2', 'dominant_color_3', 'dominant_color_4',
       'dominant_color_5', 'color_palette_json', 'extraction_status',
       'sentiment_polarity', 'sentiment_subjectivity', 'top_words',
       'top_keywords', 'cluster', 'strategy', 'strategy_group'],
      dtype='object')


## 3. Create Documents for RAG

rebuild documents

In [ ]:
def row_to_document(row):
    parts = []

    industry = row.get("competitor", "")
    if pd.notna(industry) and str(industry).strip():
        parts.append(f"Industry: {industry}")

    category = row.get("all_categories_full", "")
    if pd.notna(category) and str(category).strip():
        parts.append(f"Category: {category}")

    strategy = row.get("strategy", "")
    if pd.notna(strategy) and str(strategy).strip():
        parts.append(f"Strategy: {strategy}")

    strategy_group = row.get("strategy_group", "")
    if pd.notna(strategy_group) and str(strategy_group).strip():
        parts.append(f"Strategy group: {strategy_group}")

    sentiment = row.get("sentiment_polarity", "")
    if pd.notna(sentiment):
        parts.append(f"Sentiment score: {sentiment}")

    keywords = row.get("top_keywords", "")
    if pd.notna(keywords) and str(keywords).strip():
        parts.append(f"Keywords: {keywords}")

    ocr = row.get("ocr_text", "")
    if pd.notna(ocr) and str(ocr).strip():
        parts.append(f"Ad text: {str(ocr)[:400]}")

    layout = row.get("layout_type", "")
    if pd.notna(layout) and str(layout).strip():
        parts.append(f"Layout type: {layout}")

    return " | ".join(parts)

In [ ]:
documents = [row_to_document(ads.iloc[i]) for i in range(len(ads))]
print("Created", len(documents), "documents")
print(documents[0])

Created 2046 documents
Industry: chips | Category: Chips, snacks, nuts, fruit, gum, cereal, yogurt, soups|Chocolate, cookies, candy, ice cream | Strategy: Food, Lifestyle & Consumer Branding | Strategy group: Brand Storytelling | Sentiment score: 0.4033 | Keywords: flavors|fine|eating | Ad text: For she's eating Necco Wafers. COOL, CRISP, REFRESHING DISKS OF MANY FINE FLAVORS IN EACH ASSORTED ROLL. | Layout type: image_heavy


## 4. Build Vector Store (Embeddings)

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding documents...")
embeddings = embed_model.encode(documents, show_progress_bar=True)
embeddings = np.array(embeddings).astype("float32")

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print(f"Indexed {index.ntotal} documents in FAISS")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding documents...


Batches:   0%|          | 0/64 [00:00<?, ?it/s]

Indexed 2046 documents in FAISS


In [ ]:
def retrieve_docs(question, top_k=3):

    # embed question
    q_embedding = embed_model.encode([question])
    q_embedding = np.array(q_embedding).astype("float32")

    # FAISS search
    distances, indices = index.search(q_embedding, top_k)

    results = [documents[i] for i in indices[0]]

    return results

In [ ]:
print(retrieve_docs("beauty brand strategy", top_k=2))

['Industry: beauty | Category: Beauty products and cosmetics (deodorants, toothpaste, makeup, hair products, laser hair removal, etc.)|Self esteem, bullying, cyber bullying|Cleaning products (detergents, fabric softeners, soap, tissues, paper towels, etc.) | Strategy: Food, Lifestyle & Consumer Branding | Strategy group: Brand Storytelling | Sentiment score: 0.3375 | Keywords: uk|join|dove | Ad text: L] flawed? _] flawless? |s beautiful skin only ever spotless? Join the beauty debate, campaignforrealbeauty.co.uk | Dove | Layout type: image_heavy', 'Industry: beauty | Category: Beauty products and cosmetics (deodorants, toothpaste, makeup, hair products, laser hair removal, etc.)|Cleaning products (detergents, fabric softeners, soap, tissues, paper towels, etc.) | Strategy: Food, Lifestyle & Consumer Branding | Strategy group: Brand Storytelling | Sentiment score: 0.3375 | Keywords: uk|join|dove | Ad text: flawed? _] flawless? ls beautiful skin only ever spotless? Join the beauty debate

## 5. Load Qwen LLM

In [ ]:
import json

INSIGHT_DIR = "/content/drive/MyDrive/Colab Notebooks/intelligence_ai_marketing/analytics/insights"

with open(f"{INSIGHT_DIR}/strategy_insights.json") as f:
    strategy_insights = json.load(f)

with open(f"{INSIGHT_DIR}/brand_strategy_patterns.json") as f:
    brand_patterns = json.load(f)

with open(f"{INSIGHT_DIR}/visual_design_patterns.json") as f:
    visual_patterns = json.load(f)

print("Analytics insights loaded")

Analytics insights loaded


In [ ]:
!pip install -q transformers accelerate bitsandbytes

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "Qwen/Qwen2.5-3B-Instruct"

# 4bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

## 6. RAG Query Function

In [ ]:
def clean_response(text):
    markers = [
        "DOMINANT STRATEGY:",
        "CURRENT PATTERN:",
        "AD CONCEPT"
    ]

    # if main title appears twice, keep the first only
    for marker in markers:
        first = text.find(marker)
        if first != -1:
            second = text.find(marker, first + len(marker))
            if second != -1:
                text = text[:second].strip()

    return text.strip()

In [ ]:
# Retrieve
def debug_retrieve(question, top_k=2):
    docs = retrieve_docs(question, top_k=top_k)
    for i, d in enumerate(docs, 1):
        print(f"\n--- Retrieved Doc {i} ---\n")
        print(d[:800])

def run_rag(question: str, task_prompt: str, top_k: int = 3, max_new_tokens: int = 160):
    import torch
    torch.cuda.empty_cache()

    docs = retrieve_docs(question, top_k=top_k)
    docs = [d[:450] for d in docs]
    context = "\n\n---\n\n".join(docs)

    prompt = f"""
You are a senior marketing strategy consultant.

You have access to a structured advertising dataset.

==============================
DATASET INSIGHTS
==============================

Strategy patterns:
{strategy_insights}

Brand strategy patterns:
{brand_patterns}

Visual design patterns:
{visual_patterns}

==============================
RELEVANT ADS
==============================

{context}

==============================
USER QUESTION
==============================

{question}

==============================
TASK
==============================

{task_prompt}
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return clean_response(response.strip())

test

In [ ]:
print(retrieve_docs("beauty brand strategy", top_k=2))

['Industry: beauty | Category: Beauty products and cosmetics (deodorants, toothpaste, makeup, hair products, laser hair removal, etc.)|Self esteem, bullying, cyber bullying|Cleaning products (detergents, fabric softeners, soap, tissues, paper towels, etc.) | Strategy: Food, Lifestyle & Consumer Branding | Strategy group: Brand Storytelling | Sentiment score: 0.3375 | Keywords: uk|join|dove | Ad text: L] flawed? _] flawless? |s beautiful skin only ever spotless? Join the beauty debate, campaignforrealbeauty.co.uk | Dove | Layout type: image_heavy', 'Industry: beauty | Category: Beauty products and cosmetics (deodorants, toothpaste, makeup, hair products, laser hair removal, etc.)|Cleaning products (detergents, fabric softeners, soap, tissues, paper towels, etc.) | Strategy: Food, Lifestyle & Consumer Branding | Strategy group: Brand Storytelling | Sentiment score: 0.3375 | Keywords: uk|join|dove | Ad text: flawed? _] flawless? ls beautiful skin only ever spotless? Join the beauty debate

## 7. Run Queries

1. marketing strategy analysis function

In [ ]:
def analyze_industry_strategy(industry):
    question = f"What marketing strategies are commonly used in the {industry} industry?"

    task_prompt = """
Answer like a marketing strategy consultant.

Use this structure exactly once:

MARKET PATTERN
Describe the main pattern in the dataset.

STRATEGIC INSIGHT
Explain what this means for the industry.

COMPETITIVE GAP
Identify what competitors may be underusing or missing.

RECOMMENDATION
Provide a practical recommendation.

Do not repeat any section.
Do not add hashtags.
"""

    return run_rag(question, task_prompt, top_k=3, max_new_tokens=500)

In [ ]:
print(analyze_industry_strategy("beauty"))

2. find the gaps we have from competitors. where did they miss?

In [ ]:
def find_strategy_gap(industry):
    question = f"What opportunities are competitors in the {industry} industry missing, and how could a new brand differentiate?"

    task_prompt = """
Answer like a strategy consultant.

Use exactly this structure:

CURRENT PATTERN
What are competitors mostly doing now?

GAP
What are they underusing or missing?

RECOMMENDATION
How should a new brand differentiate?

Keep the answer under 180 words.
Be practical and specific.
"""

    return run_rag(question, task_prompt, top_k=3, max_new_tokens=500)

In [ ]:
print(find_strategy_gap("beauty"))

3. Ideas on New Ads; Creative Generation

In [ ]:
def generate_ad_concept(industry, target_audience, style):
    question = f"""
Generate an advertising concept for a {industry} brand targeting {target_audience}.
Use a {style} style.
"""

    task_prompt = """
Generate a creative ad concept based on the dataset patterns.

Use exactly this structure:

HEADLINE
One short ad headline.

CORE MESSAGE
One or two sentences.

VISUAL CONCEPT
Describe the visual direction.

CTA
One short call-to-action.

Keep it concise and concrete.
"""

    return run_rag(question, task_prompt, top_k=3, max_new_tokens=500)

In [ ]:
print(generate_ad_concept("beauty", "Gen Z", "minimal and elegant"))

4. get reference ads

In [ ]:
def get_reference_ads(industry):
    docs = retrieve_docs(f"{industry} advertising strategy", top_k=5)

    unique_docs = []
    seen_prefix = set()

    for d in docs:
        key = d[:180].strip()
        if key not in seen_prefix:
            seen_prefix.add(key)
            unique_docs.append(d)

        if len(unique_docs) == 2:
            break

    return "\n\n---\n\n".join(unique_docs)

In [ ]:
import re

# =========================
# OUTPUT CLEANERS
# =========================

def clean_consulting_output(text):
    lines = text.splitlines()
    cleaned_lines = []

    for line in lines:
        stripped = line.strip()

        if stripped.startswith("Do not add"):
            continue
        if stripped.startswith("Stop after"):
            continue
        if stripped.startswith("Keep the answer"):
            continue
        if stripped.startswith("Based on the provided dataset"):
            continue

        cleaned_lines.append(line)

    return "\n".join(cleaned_lines).strip()


def extract_consulting_sections(text):
    sections = ["MARKET PATTERN", "STRATEGIC INSIGHT", "COMPETITIVE GAP", "RECOMMENDATION"]
    result = []

    for i, sec in enumerate(sections):
        next_sections = sections[i+1:]
        if next_sections:
            pattern = rf"{sec}\s*(.*?)(?={'|'.join(next_sections)}|$)"
        else:
            pattern = rf"{sec}\s*(.*)$"

        match = re.search(pattern, text, flags=re.DOTALL | re.IGNORECASE)
        if match:
            content = match.group(1).strip()
            content = re.sub(r"\s+", " ", content)
            result.append(f"{sec}\n{content}")

    return "\n\n".join(result).strip()


def extract_gap_sections(text):
    sections = ["CURRENT PATTERN", "GAP", "RECOMMENDATION"]
    result = []

    for i, sec in enumerate(sections):
        next_sections = sections[i+1:]
        if next_sections:
            pattern = rf"{sec}\s*(.*?)(?={'|'.join(next_sections)}|$)"
        else:
            pattern = rf"{sec}\s*(.*)$"

        match = re.search(pattern, text, flags=re.DOTALL | re.IGNORECASE)
        if match:
            content = match.group(1).strip()
            content = re.sub(r"\s+", " ", content)
            result.append(f"{sec}\n{content}")

    return "\n\n".join(result).strip()


def extract_ad_sections(text):
    sections = ["Headline", "Core Message", "Visual Concept", "CTA"]
    result = []

    for i, sec in enumerate(sections):
        next_sections = sections[i+1:]
        if next_sections:
            pattern = rf"{sec}:\s*(.*?)(?={'|'.join([s + ':' for s in next_sections])}|$)"
        else:
            pattern = rf"{sec}:\s*(.*)$"

        match = re.search(pattern, text, flags=re.DOTALL | re.IGNORECASE)
        if match:
            content = match.group(1).strip()
            content = re.sub(r"\s+", " ", content)
            result.append(f"{sec}:\n{content}")

    return "\n\n".join(result).strip()


# =========================
# FINAL RAG WRAPPER
# =========================

def run_rag(question: str, task_prompt: str, top_k: int = 2, max_new_tokens: int = 180, temperature: float = 0.3):
    import torch
    torch.cuda.empty_cache()

    docs = retrieve_docs(question, top_k=top_k)
    docs = [d[:450] for d in docs]
    context = "\n\n---\n\n".join(docs)

    prompt = f"""
You are a senior marketing strategy consultant.

You have access to a structured advertising dataset.

==============================
DATASET INSIGHTS
==============================

Strategy patterns:
{strategy_insights}

Brand strategy patterns:
{brand_patterns}

Visual design patterns:
{visual_patterns}

==============================
RELEVANT ADS
==============================

{context}

==============================
USER QUESTION
==============================

{question}

==============================
TASK
==============================

{task_prompt}
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return clean_consulting_output(response.strip())


# =========================
# FINAL FEATURES
# =========================

def analyze_industry_strategy(industry):
    question = f"What marketing strategies are commonly used in the {industry} industry?"

    task_prompt = """
Answer like a marketing strategy consultant.

Use this structure exactly once:

MARKET PATTERN
Describe the main pattern in the dataset.

STRATEGIC INSIGHT
Explain what this means for the industry.

COMPETITIVE GAP
Identify what competitors may be underusing or missing.

RECOMMENDATION
Provide a practical recommendation.

Do not repeat any section.
Do not add hashtags.
"""

    raw = run_rag(question, task_prompt, top_k=2, max_new_tokens=300, temperature=0.3)
    return extract_consulting_sections(raw)


def find_strategy_gap(industry):
    question = f"What opportunities are competitors in the {industry} industry missing, and how could a new brand differentiate?"

    task_prompt = """
Answer like a strategy consultant.

Use this structure exactly once:

CURRENT PATTERN
What are competitors mostly doing now?

GAP
What are they underusing or missing?

RECOMMENDATION
How should a new brand differentiate?

Do not repeat any section.
Do not add hashtags.
"""

    raw = run_rag(question, task_prompt, top_k=2, max_new_tokens=300, temperature=0.3)
    return extract_gap_sections(raw)


def generate_ad_concept(industry, target_audience, style):
    question = f"""
Generate an advertising concept for a {industry} brand targeting {target_audience}.
Use a {style} style.
"""

    task_prompt = """
Generate a creative advertising concept.

Return ONLY the following structure:

Headline:
Core Message:
Visual Concept:
CTA:

Rules:
- Do not repeat the structure.
- Do not add explanations.
- Do not add emojis.
- Do not add extra paragraphs.
- Stop after CTA.
"""

    raw = run_rag(question, task_prompt, top_k=2, max_new_tokens=170, temperature=0.6)
    return extract_ad_sections(raw)


def get_reference_ads(industry):
    docs = retrieve_docs(f"{industry} advertising strategy", top_k=2)
    return "\n\n---\n\n".join(docs)

ui试运行

In [ ]:
import gradio as gr
import traceback

def ui_analyze_strategy(industry, audience, style):
    try:
        return analyze_industry_strategy(industry)
    except Exception:
        return "ERROR:\n" + traceback.format_exc()

def ui_find_gap(industry, audience, style):
    try:
        return find_strategy_gap(industry)
    except Exception:
        return "ERROR:\n" + traceback.format_exc()

def ui_generate_concept(industry, audience, style):
    try:
        return generate_ad_concept(industry, audience, style)
    except Exception:
        return "ERROR:\n" + traceback.format_exc()

def ui_show_refs(industry, audience, style):
    try:
        return get_reference_ads(industry)
    except Exception:
        return "ERROR:\n" + traceback.format_exc()

with gr.Blocks() as demo:
    gr.Markdown("# AI Marketing Strategy Assistant")
    gr.Markdown(
        "Use the assistant step by step to analyze strategy, identify gaps, "
        "generate ad concepts, and inspect reference ads."
    )

    with gr.Row():
        industry = gr.Textbox(label="Industry", value="beauty")
        audience = gr.Textbox(label="Target Audience", value="Gen Z")
        style = gr.Textbox(label="Campaign Style", value="minimal and elegant")

    with gr.Row():
        btn_strategy = gr.Button("Analyze Strategy")
        btn_gap = gr.Button("Find Competitive Gap")
        btn_concept = gr.Button("Generate Ad Concept")
        btn_refs = gr.Button("Show Reference Ads")

    gr.Markdown("## Outputs")

    strategy_output = gr.Textbox(label="Strategy Analysis", lines=10)
    gap_output = gr.Textbox(label="Competitive Gap", lines=10)
    concept_output = gr.Textbox(label="Ad Concept", lines=10)
    refs_output = gr.Textbox(label="Reference Ads", lines=12)

    btn_strategy.click(
        fn=ui_analyze_strategy,
        inputs=[industry, audience, style],
        outputs=strategy_output
    )

    btn_gap.click(
        fn=ui_find_gap,
        inputs=[industry, audience, style],
        outputs=gap_output
    )

    btn_concept.click(
        fn=ui_generate_concept,
        inputs=[industry, audience, style],
        outputs=concept_output
    )

    btn_refs.click(
        fn=ui_show_refs,
        inputs=[industry, audience, style],
        outputs=refs_output
    )

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d93bb4bafae74a6be0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
